# Executive Dashboard Dataset
## Purpose: Create executive-level inventory KPIs for reporting and visualization.

In [1]:
#  Connect to Database

import pandas as pd
import sqlite3

db_path = r"C:\Users\bturn\PycharmProjects\bturner\Inventory_Flow_Command_Center\inventory_flow.db"

conn = sqlite3.connect(db_path)

In [2]:
# Inventory Health by Region

inventory_health = pd.read_sql("""
SELECT
    Region,
    AVG("Inventory Level") AS Avg_Inventory,
    SUM("Units Sold") AS Total_Units_Sold,
    SUM("Units Ordered") AS Total_Units_Ordered
FROM fact_inventory
GROUP BY Region
ORDER BY Total_Units_Sold DESC;
""", conn)

inventory_health


,Region,Avg_Inventory,Total_Units_Sold,Total_Units_Ordered
0,East,275.292659,2511265,2017919
1,South,275.313221,2507799,2015826
2,North,274.796632,2484966,1997677
3,West,272.468122,2471552,2009905


In [3]:
# Stockout Rate

stockout_rate = pd.read_sql("""
SELECT
    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Inventory Level" = 0 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS Stockout_Rate_Pct
FROM fact_inventory;
""", conn)

stockout_rate

,Stockout_Rate_Pct
0,0.0


In [4]:
# Days of Supply

days_supply = pd.read_sql("""
SELECT
    ROUND(
        AVG(
            CAST("Inventory Level" AS FLOAT)
            / NULLIF("Units Sold",0)
        ),
        2
    ) AS Avg_Days_Of_Supply
FROM fact_inventory;
""", conn)

days_supply

,Avg_Days_Of_Supply
0,6.17


In [5]:
# Top Risk Stores

risk_stores = pd.read_sql("""
SELECT
    "Store ID",
    AVG("Inventory Level") AS Avg_Inventory,
    SUM("Units Sold") AS Units_Sold
FROM fact_inventory
GROUP BY "Store ID"
ORDER BY Avg_Inventory ASC
LIMIT 10;
""", conn)

risk_stores

,Store ID,Avg_Inventory,Units_Sold
0,S004,272.304788,1979245
1,S002,273.327907,1987715
2,S001,273.339945,1975750
3,S005,276.504446,2010176
4,S003,276.872298,2022696


In [6]:
# Executive Dashboard Dataset

dashboard_df = pd.read_sql("""
SELECT
    Region,
    Category,
    COUNT(DISTINCT "Store ID") AS Stores,
    AVG("Inventory Level") AS Avg_Inventory,
    SUM("Units Sold") AS Units_Sold,
    SUM("Units Ordered") AS Units_Ordered,
    SUM(
        CASE
            WHEN "Inventory Level" = 0
            THEN 1
            ELSE 0
        END
    ) AS Stockout_Events
FROM fact_inventory
GROUP BY
    Region,
    Category
""", conn)

dashboard_df.head()

,Region,Category,Stores,Avg_Inventory,Units_Sold,Units_Ordered,Stockout_Events
0,East,Clothing,5,272.649614,507229,416908,0
1,East,Electronics,5,273.079892,500086,405723,0
2,East,Furniture,5,276.137233,500612,393991,0
3,East,Groceries,5,279.935939,513781,409460,0
4,East,Toys,5,274.727500,489557,391837,0


In [7]:
# Export for Power BI

dashboard_df.to_csv(
    "executive_dashboard_dataset.csv",
    index=False
)

print("Executive dashboard dataset created.")

Executive dashboard dataset created.
